# Notebook 3: Missing Value Handling


In [1]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "Age": [22, np.nan, 28, 35, np.nan, 40],
    "Salary": [25000, 30000, np.nan, 45000, 50000, np.nan],
    "City": ["Bengaluru", "Mysuru", np.nan, "Hassan", "Bengaluru", np.nan],
    "Department": ["IT", "HR", "IT", "HR", "IT", "HR"]
})
print(df)

    Age   Salary       City Department
0  22.0  25000.0  Bengaluru         IT
1   NaN  30000.0     Mysuru         HR
2  28.0      NaN        NaN         IT
3  35.0  45000.0     Hassan         HR
4   NaN  50000.0  Bengaluru         IT
5  40.0      NaN        NaN         HR


## 1. What are Missing Values?

### Definition
Missing values are entries where information is absent or unknown. In Pandas, they commonly appear as NaN, None, or NaT. Blank strings and special codes such as "NA" may also mean missing data. Missing values must be recognized before analysis.

### Business Example
A customer registration form may have a blank age or salary field. The business cannot assume the blank value is zero. Analysts first check how many fields are incomplete. They then decide whether to recover, retain, or estimate the values.

### AI/ML Use
Missing features may cause some ML algorithms to fail. They can also distort summary statistics and predictions. A missingness indicator may itself contain useful information. The correct strategy depends on the feature and missingness mechanism.

### Python Example

In [3]:
print(df.isna())

     Age  Salary   City  Department
0  False   False  False       False
1   True   False  False       False
2  False    True   True       False
3  False   False  False       False
4   True   False  False       False
5  False    True   True       False


### Explanation
The `isna()` method marks missing entries as True. It helps locate gaps before choosing a treatment.

### When to Use
Use when checking any new dataset.

### When Not to Use
Do not assume all blank-looking strings are automatically NaN.

### Advantages
Quick, built-in detection.

### Limitations
Cannot explain why a value is missing.

## 2. Why Missing Values Occur

### Definition
Missing values can arise from skipped forms, broken sensors, merging datasets, or unavailable records. Sometimes a question does not apply to a person. Missingness can be random or related to observed or hidden information. Understanding the cause is important before filling data.

### Business Example
An online shop may lack customer income because it is optional. A sensor may fail during a power cut. These two cases have different causes. A business should investigate collection processes rather than immediately fill blanks.

### AI/ML Use
Missingness can introduce bias into training data. The model may learn an incorrect relationship if the cause is ignored. Domain knowledge helps choose imputation or exclusion. Missing indicators can sometimes improve predictions.

### Python Example

In [5]:
print(df.isna().sum())

Age           2
Salary        2
City          2
Department    0
dtype: int64


### Explanation
Counts reveal which fields need investigation. The count alone does not reveal the missingness mechanism.

### When to Use
Use during initial data-quality investigation.

### When Not to Use
Do not conclude that missingness is random from counts alone.

### Advantages
Highlights affected columns.

### Limitations
Does not identify the root cause.

## 3. MCAR – Missing Completely At Random

### Definition
MCAR means the chance of a value being missing is unrelated to observed or unobserved data. For example, a few forms may be lost accidentally. This is a strong statistical assumption. It cannot be established by a simple null count.

### Business Example
A random printer fault destroys a few customer forms. The fault is unrelated to customer income or age. This is a possible MCAR situation. The company should still check whether the fault was truly random.

### AI/ML Use
Under MCAR, deleting a small number of incomplete rows may preserve representativeness. However, it reduces sample size. Statistical tests and domain investigation can help assess the assumption. MCAR should not be claimed from a toy example.

### Python Example

In [7]:
sample = pd.Series([10, np.nan, 30, 40])
print(sample.isna())

0    False
1     True
2    False
3    False
dtype: bool


### Explanation
The code identifies missing positions only. It does not prove that the missingness is MCAR.

### When to Use
Use the MCAR concept when missingness is plausibly independent of all values.

### When Not to Use
Do not label data MCAR solely because missing values look scattered.

### Advantages
Can justify simple deletion under appropriate conditions.

### Limitations
The assumption is difficult to verify fully.

## 4. MAR – Missing At Random

### Definition
MAR means missingness can depend on observed variables but not on the missing value after accounting for those observed variables. For example, income may be missing more often for a recorded age group. MAR does not mean values disappear randomly without a pattern. Observed features can help estimate missing entries.

### Business Example
Customers in a particular age group may skip an income question more often. Their age is recorded even when income is not. The company can use age and other known features to understand the pattern. This supports a conditional imputation approach.

### AI/ML Use
Multivariable imputation may be suitable under a plausible MAR assumption. Other observed features can help predict missing entries. Analysts should compare missingness across known groups. The assumption still requires careful judgment.

### Python Example

In [9]:
temp = pd.DataFrame({"Group": ["A", "A", "B", "B"], "Income": [20000, np.nan, 40000, np.nan]})
print(temp.groupby("Group")["Income"].apply(lambda x: x.isna().mean()))

Group
A    0.5
B    0.5
Name: Income, dtype: float64


### Explanation
The code shows missing rates by an observed group. Differences may suggest a relationship, but they do not prove MAR.

### When to Use
Use when observed variables can explain missingness.

### When Not to Use
Do not assume MAR merely because a group difference appears.

### Advantages
Allows observed features to inform imputation.

### Limitations
Depends on whether important explanatory variables were recorded.

## 5. MNAR – Missing Not At Random

### Definition
MNAR means the probability of missingness depends on the missing value itself, even after considering observed data. For example, people with very high incomes may avoid reporting income. This can create systematic bias. Ordinary imputation may not correct that bias.

### Business Example
High-income customers may be less willing to share their salaries. Their missing salaries are not representative of all salaries. A simple mean can underestimate high incomes. The business may need better collection methods or sensitivity analysis.

### AI/ML Use
MNAR can lead to biased models even after filling missing entries. External information and domain expertise may be necessary. Analysts may compare results under multiple assumptions. A missingness flag alone does not solve MNAR.

### Python Example

In [11]:
income = pd.Series([25000, 30000, np.nan, 45000])
print("Missing incomes:", income.isna().sum())

Missing incomes: 1


### Explanation
The code counts absent incomes. It cannot determine whether high earners withheld their values.

### When to Use
Use the MNAR concept when missingness plausibly depends on the unseen value.

### When Not to Use
Do not treat mean imputation as an unbiased fix for MNAR.

### Advantages
Encourages realistic assessment of systematic missingness.

### Limitations
The missing values are unobserved, making MNAR hard to confirm.

## 6. Detecting Missing Values

### Definition
Detecting missing values is the first practical cleaning step. Pandas provides `isna()` and `isnull()` for this task. They identify standard missing markers such as NaN. Custom placeholders must be converted to missing markers first.

### Business Example
A sales sheet may contain blank prices and the text "unknown". A report should identify both forms of missing data. Otherwise some gaps remain hidden. The analyst standardizes missing markers before further checks.

### AI/ML Use
Models need consistent handling of null values. Detecting gaps helps avoid unexpected training errors. It also guides feature-specific preprocessing. Placeholder strings should not be treated as real categories automatically.

### Python Example

In [13]:
print(df.isna().sum())

Age           2
Salary        2
City          2
Department    0
dtype: int64


### Explanation
This prints the number of missing entries in each column.

### When to Use
Use at the start of data inspection.

### When Not to Use
Do not rely on it to detect unstandardized placeholders like "?".

### Advantages
Fast and easy to interpret.

### Limitations
Requires extra cleanup for nonstandard markers.

## 7. Missing Value Percentage

### Definition
Missing value percentage expresses absent entries as a share of total rows. It helps compare columns of different sizes. A high percentage may signal an unreliable feature. There is no universal percentage threshold for deletion.

### Business Example
An insurance company compares missing age and missing claim amount. Age may be missing in 5% of records and claim amount in 45%. These rates guide further investigation. The importance of each field still matters.

### AI/ML Use
Missing percentages help prioritize preprocessing. A feature with many gaps may still be useful. The model goal and missingness pattern matter more than a fixed cutoff. Missingness should also be checked across subgroups.

### Python Example

In [15]:
print((df.isna().mean() * 100).round(1))

Age           33.3
Salary        33.3
City          33.3
Department     0.0
dtype: float64


### Explanation
`isna().mean()` calculates the missing fraction in each column. Multiplying by 100 gives a percentage.

### When to Use
Use when assessing completeness by column.

### When Not to Use
Do not drop columns automatically at an arbitrary percentage.

### Advantages
Makes missingness easy to compare.

### Limitations
Does not describe why or where values are missing.

## 8. Dropping Rows

### Definition
Dropping rows removes observations containing missing values. Pandas uses `dropna()` for this operation. It is simple but reduces the available sample. It can also create bias if incomplete rows differ systematically.

### Business Example
A shop may remove a few unusable purchase records. If only a tiny random portion is incomplete, this may be acceptable. If many customers are affected, important patterns could be lost. The shop should check how many records disappear.

### AI/ML Use
Complete-case analysis can work when missingness is limited and assumptions are appropriate. Dropping rows reduces training examples. It may change class balance. Always compare row counts before and after.

### Python Example

In [17]:
print(df.dropna())

    Age   Salary       City Department
0  22.0  25000.0  Bengaluru         IT
3  35.0  45000.0     Hassan         HR


### Explanation
`dropna()` returns only rows with no missing values in any column.

### When to Use
Use when few rows are missing and their removal is defensible.

### When Not to Use
Avoid when missingness is extensive or systematically related to outcomes.

### Advantages
Simple; does not invent values.

### Limitations
Loses data and can bias results.

## 9. Dropping Columns

### Definition
Dropping columns removes entire features with missing values. It can be sensible when a feature has little value or is mostly unavailable. The decision should consider predictive importance and domain meaning. A high missing rate alone is not enough.

### Business Example
A survey may contain an optional field that almost nobody answers. If it has little business value, the company may exclude it. A critical risk variable should not be discarded so easily. Feature usefulness should be assessed first.

### AI/ML Use
Removing a weak feature can simplify a model. Removing an informative feature can reduce accuracy. Decisions should be evaluated on validation data. Feature selection should avoid leaking test-set information.

### Python Example

In [19]:
print(df.drop(columns=["Salary"]).head())

    Age       City Department
0  22.0  Bengaluru         IT
1   NaN     Mysuru         HR
2  28.0        NaN         IT
3  35.0     Hassan         HR
4   NaN  Bengaluru         IT


### Explanation
The example removes the Salary column and keeps the other features.

### When to Use
Use when a feature is unusable or unimportant.

### When Not to Use
Avoid dropping important features solely due to missingness.

### Advantages
Reduces complexity.

### Limitations
Can discard valuable predictive information.

## 10. Mean Imputation

### Definition
Mean imputation replaces missing numerical values with the average of observed values. It is easy and fast. It is most reasonable for roughly symmetric data without strong outliers. It can shrink variance and weaken relationships.

### Business Example
A company has a few missing employee ages. If ages are fairly balanced, the mean may be a simple baseline. If the data contain extreme values, the mean may be misleading. Analysts should inspect the distribution first.

### AI/ML Use
Mean imputation can provide a baseline for numeric ML features. The mean must be learned from training data only. It does not preserve the original distribution perfectly. Compare it with more appropriate alternatives.

### Python Example

In [21]:
print(df["Age"].fillna(df["Age"].mean()))

0    22.00
1    31.25
2    28.00
3    35.00
4    31.25
5    40.00
Name: Age, dtype: float64


### Explanation
The mean of known ages replaces missing ages. The original `df` remains unchanged.

### When to Use
Use for low missingness in approximately symmetric numeric features.

### When Not to Use
Avoid for highly skewed data, extreme outliers, or meaningful missingness patterns.

### Advantages
Fast and easy.

### Limitations
Reduces variance and can distort correlations.

## 11. Median Imputation

### Definition
Median imputation replaces missing numerical values with the middle observed value. The median is less affected by extreme values than the mean. It is useful for skewed distributions. It still hides some variation in the data.

### Business Example
Customer spending often has a few very large purchases. The average can be pulled upward by these transactions. The median better represents a typical customer. Missing spending values can use the median as a baseline.

### AI/ML Use
Median imputation is common for skewed numerical features. It is robust to outliers. Fit the median using training data only. It should still be validated against other methods.

### Python Example

In [23]:
print(df["Salary"].fillna(df["Salary"].median()))

0    25000.0
1    30000.0
2    37500.0
3    45000.0
4    50000.0
5    37500.0
Name: Salary, dtype: float64


### Explanation
Missing salaries are replaced with the median of observed salaries.

### When to Use
Use for skewed numeric data or data with outliers.

### When Not to Use
Avoid when missingness depends strongly on other features or unseen values.

### Advantages
Resistant to extreme values.

### Limitations
Can reduce variation and ignore feature relationships.

## 12. Mode Imputation

### Definition
Mode imputation replaces missing values with the most frequent observed value. It is commonly used for categorical variables. It can also work for discrete numeric features. It may overrepresent the dominant category.

### Business Example
A retail dataset contains missing city names. If one city occurs most often, it can be used as a simple baseline. However, this may exaggerate that city’s share. An explicit "Unknown" category might sometimes be better.

### AI/ML Use
Mode imputation helps many algorithms process categorical data. It is easy to implement in a pipeline. It can worsen class or category imbalance. Training data should determine the mode.

### Python Example

In [25]:
print(df["City"].fillna(df["City"].mode()[0]))

0    Bengaluru
1       Mysuru
2    Bengaluru
3       Hassan
4    Bengaluru
5    Bengaluru
Name: City, dtype: object


### Explanation
The first mode is used to replace missing city names.

### When to Use
Use for categorical variables when a frequent category is a sensible guess.

### When Not to Use
Avoid when unknown itself is informative or categories are sensitive to distortion.

### Advantages
Simple and compatible with categorical data.

### Limitations
Can inflate the most common category.

## 13. Constant Value Imputation

### Definition
Constant imputation replaces missing entries with a chosen fixed value. Common examples are "Unknown" for categories or a sentinel for numeric data. The constant must have a clear meaning. Numeric sentinels can distort scales and statistics.

### Business Example
An online form may not record a customer’s preferred language. The company can label it "Unknown" rather than guessing. This keeps missingness visible. Analysts should distinguish unknown from an actual category.

### AI/ML Use
Constant imputation can preserve information about missingness. It often works well for categorical fields. Numeric sentinels need careful handling and possibly a missingness flag. Some models may misinterpret unusual sentinel values.

### Python Example

In [27]:
print(df["City"].fillna("Unknown"))

0    Bengaluru
1       Mysuru
2      Unknown
3       Hassan
4    Bengaluru
5      Unknown
Name: City, dtype: object


### Explanation
Missing cities are explicitly labeled Unknown.

### When to Use
Use when missingness should remain identifiable, especially in categorical fields.

### When Not to Use
Avoid arbitrary numeric constants that look like real measurements.

### Advantages
Transparent and easy to apply.

### Limitations
Creates a new category or artificial numeric values.

## 14. Forward Fill

### Definition
Forward fill copies the last available value into the next missing position. It is used for ordered observations, especially time series. Row order is essential. It assumes the previous value remains applicable.

### Business Example
A warehouse records stock levels every hour. If one hourly reading is missing, the last known level may be a temporary estimate. This can work if stock changes slowly. It is unsafe when large changes may occur between readings.

### AI/ML Use
Forward fill can support time-series preprocessing. The data must be sorted chronologically. It should not carry values across unrelated customers or groups. It can hide sudden changes.

### Python Example

In [29]:
s = pd.Series([10, np.nan, 12, np.nan])
print(s.ffill())

0    10.0
1    10.0
2    12.0
3    12.0
dtype: float64


### Explanation
Each missing value receives the most recent earlier observation.

### When to Use
Use for ordered or time-series data when the previous value remains valid.

### When Not to Use
Avoid on unordered rows or across unrelated entities.

### Advantages
Very simple; maintains continuity.

### Limitations
Can propagate stale values and leaves leading gaps.

## 15. Backward Fill

### Definition
Backward fill copies the next available value into an earlier missing position. It depends on a meaningful order. It assumes the next observation is representative of the gap. It may introduce future information into past records.

### Business Example
A historical sensor record has a short missing interval. The next reading may help reconstruct an offline report. However, a live prediction system cannot know future readings. The company must consider when the data are available.

### AI/ML Use
Backward fill may be suitable for retrospective time-series analysis. It is usually inappropriate for causal or real-time prediction if it uses future values. Always check leakage risk. Avoid filling across unrelated groups.

### Python Example

In [31]:
s = pd.Series([np.nan, 10, np.nan, 12])
print(s.bfill())

0    10.0
1    10.0
2    12.0
3    12.0
dtype: float64


### Explanation
Each missing value receives the next available observation.

### When to Use
Use for retrospective ordered data when future observations are legitimately available.

### When Not to Use
Avoid for real-time forecasting or when future values would leak information.

### Advantages
Simple; can fill leading gaps.

### Limitations
May leak future information and leaves trailing gaps.

## 16. Interpolation

### Definition
Interpolation estimates missing numerical values using nearby known observations. Linear interpolation assumes gradual change between points. It is especially useful for evenly spaced time series. It is not suitable for every variable.

### Business Example
A temperature sensor misses a reading between 25°C and 27°C. Linear interpolation estimates an intermediate temperature. This is reasonable if the temperature changes smoothly. It is less reliable during sudden spikes.

### AI/ML Use
Interpolation can maintain continuity in numerical sequences. The method and time spacing matter. It can smooth out genuine extremes. It should not be used on unordered categorical records.

### Python Example

In [33]:
s = pd.Series([10, np.nan, 14])
print(s.interpolate())

0    10.0
1    12.0
2    14.0
dtype: float64


### Explanation
Linear interpolation estimates the middle value as 12.

### When to Use
Use for continuous numeric sequences with a plausible smooth trend.

### When Not to Use
Avoid for categorical data, abrupt jumps, or unordered observations.

### Advantages
Uses neighboring information.

### Limitations
Can invent misleading smooth values.

## 17. Group-Based Imputation

### Definition
Group-based imputation fills values using statistics from relevant subgroups. For example, salaries may be filled using the median within each department. It can reflect real differences between groups. Each group must contain enough valid observations.

### Business Example
IT and HR departments may have different salary ranges. Filling every salary with one company-wide mean ignores this difference. Department-specific medians can be more realistic. Small groups may need a fallback.

### AI/ML Use
Group-based imputation can preserve subgroup patterns. Group statistics must be learned from training data only. Small groups or unseen categories need careful handling. It may still miss relationships within each group.

### Python Example

In [35]:
sample = pd.DataFrame({"Team": ["IT", "IT", "HR", "HR"], "Pay": [40000, np.nan, 25000, np.nan]})
sample["Pay"] = sample["Pay"].fillna(sample.groupby("Team")["Pay"].transform("median"))
print(sample)

  Team      Pay
0   IT  40000.0
1   IT  40000.0
2   HR  25000.0
3   HR  25000.0


### Explanation
Each missing salary is filled with its own team’s median.

### When to Use
Use when reliable groups have different typical values.

### When Not to Use
Avoid for very small groups or when group labels are missing or unreliable.

### Advantages
Respects subgroup differences.

### Limitations
Needs enough data per group and careful train/test handling.

## 18. KNN Imputation

### Definition
KNN imputation estimates missing numeric values using similar records. Similarity is measured from other available features. It can capture local relationships. Results depend on feature scales and the number of neighbors.

### Business Example
A customer record has missing income but known age and spending. Similar customers may help estimate income. This may be more useful than a global average. The company must ensure the comparison features are meaningful.

### AI/ML Use
KNNImputer can use relationships across numeric features. Scaling and feature choice influence neighbors. Fit the imputer on training data only. It may be slower on large datasets.

### Python Example

In [37]:
from sklearn.impute import KNNImputer
small = pd.DataFrame({"Age": [20, 22, 40, 42], "Income": [20000, np.nan, 40000, 42000]})
result = KNNImputer(n_neighbors=2).fit_transform(small)
print(pd.DataFrame(result, columns=small.columns))

    Age   Income
0  20.0  20000.0
1  22.0  30000.0
2  40.0  40000.0
3  42.0  42000.0


### Explanation
The missing income is estimated using nearby rows based on available numeric information.

### When to Use
Use when numeric features have meaningful similarity patterns.

### When Not to Use
Avoid when features are poorly scaled, unrelated, or datasets are extremely large.

### Advantages
Uses relationships between observations.

### Limitations
Sensitive to scaling, neighbors, and computation cost.

## 19. Iterative Imputation

### Definition
Iterative imputation predicts each incomplete feature using other features. It repeats the process to improve estimates. It can model multivariable relationships. It requires more care and computation than simple imputation.

### Business Example
A bank dataset contains missing income and spending values. Other known customer attributes may help estimate both. Iterative methods update the estimates in rounds. Results still depend on assumptions and model choice.

### AI/ML Use
IterativeImputer is useful when multiple numeric features contain related missing values. It must be fitted only on training data. Convergence and model assumptions should be considered. A single imputed dataset does not fully represent uncertainty.

### Python Example

In [39]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
small = pd.DataFrame({"Age": [20, 30, 40, 50], "Income": [20000, np.nan, 40000, 50000]})
result = IterativeImputer(random_state=0, max_iter=10).fit_transform(small)
print(pd.DataFrame(result, columns=small.columns).round(1))

    Age   Income
0  20.0  20000.0
1  30.0  30000.0
2  40.0  40000.0
3  50.0  50000.0


### Explanation
The imputer estimates missing income from patterns in the other feature.

### When to Use
Use when numeric features have useful predictive relationships.

### When Not to Use
Avoid when data are too small, relationships are unreliable, or simple methods are sufficient.

### Advantages
Can use multiple feature relationships.

### Limitations
More complex, slower, and assumption-dependent.